# 🏆 [Day 37] 실전 트리플 & 온톨로지 엔드투엔드 핸즈온 워크북

> **핵심 학습 목표**:
> 비정형 의학 텍스트로부터 지식그래프의 핵심 단위인 **트리플(Triple: 주어-관계-목적어)**을 추출하고, 사전에 엄격히 정의된 **온톨로지(Ontology) 계약**에 맞춰 검증한 뒤, **표준 식별자(ID) 기반의 무결점 지식그래프**로 적재하는 전체 엔지니어링 과정을 직접 구현하고 검증합니다.
>
> 1. 📜 **[온톨로지 계약]**: 8+1대 핵심 관계 시그니처(`RELATION_SIGNATURES`)와 프롬프트 자동 생성기(`build_ontology_block`)
> 2. 🧠 **[LLM 구조화 스키마]**: Pydantic `Literal` 기반 관계/타입 Enum 강제 및 CoT 다단계 추론 모델 설계
> 3. 🔑 **[식별자(ID) 바인딩]**: 이름이 아닌 표준 식별자(`name2id.json`) 매핑과 미등록 개체 `:Candidate` 격리
> 4. 🧪 **[3단계 정제 퍼널]**: 원문 근거 대조(`ground_check`) ➔ 시그니처 일치 검증(`check_signature`) ➔ 복합키 중복 제거(`drop_duplicates`)
> 5. 🌐 **[Neo4j 멱등 적재]**: 다중 레이블 계층(`TYPE_HIERARCHY`) 및 근거 등급(`evidence_level`)을 포함한 파라미터화 Cypher MERGE 생성

## 0. 환경 설정 및 라이브러리 준비

In [1]:
import os
import sys
import json
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Literal, Set
from pydantic import BaseModel, Field

data_dir = Path('data')
assert data_dir.exists(), 'data 디렉토리가 존재하지 않습니다.'
print('✅ [환경 설정 완료] 필수 라이브러리 로드 성공')
print('✅ [작업 디렉토리]: data 디렉토리 및 의존성 확인 완료')

✅ [환경 설정 완료] 필수 라이브러리 로드 성공
✅ [작업 디렉토리]: data 디렉토리 및 의존성 확인 완료


## 1. 온톨로지 계약: 8+1대 관계 시그니처 및 프롬프트 블록 빌더

In [2]:
RELATION_SIGNATURES = {
    "TREATS": ("Compound", "Disease", "약이 질병을 치료한다. 질병의 원인이나 진행 자체에 작용한다."),
    "PALLIATES": ("Compound", "Disease", "약이 질병의 증상을 완화한다. 질병 자체는 그대로 두고 증상만 덜어 준다."),
    "BINDS": ("Compound", "Gene", "약이 그 유전자의 단백질에 결합한다. 대사·수송을 맡는다는 진술도 포함한다."),
    "UPREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 증가시킨다."),
    "DOWNREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 감소시킨다."),
    "ASSOCIATES": ("Disease", "Gene", "질병과 유전자 사이에 연관이 보고됐다."),
    "PRESENTS": ("Disease", "Symptom", "질병이 그 증상으로 나타난다."),
    "INCLUDES": ("PharmacologicClass", "Compound", "약효 분류가 그 약물을 포함한다 (계열 소속 관계)."),
    "RESEMBLES_DD": ("Disease", "Disease", "두 질병의 임상 양상이나 발병 기전이 유사하다."),
}

NODE_TYPES = {"Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"}

def build_ontology_block(signatures: dict) -> str:
    lines = ["## 허용 관계와 시그니처 (아래에 없는 관계는 절대 추출하지 마십시오)"]
    for rel, (s, o, c) in signatures.items():
        lines.append(f"- {rel}: ({s}) -> ({o}) | 판정 기준: {c}")
    return "\n".join(lines)

print(f"✅ [온톨로지 정의 완료] 총 {len(RELATION_SIGNATURES)}종 관계 시그니처 등록 완료")
print("--- [생성된 온톨로지 계약 블록] ---")
print(build_ontology_block(RELATION_SIGNATURES))

✅ [온톨로지 정의 완료] 총 9종 관계 시그니처 등록 완료
--- [생성된 온톨로지 계약 블록] ---
## 허용 관계와 시그니처 (아래에 없는 관계는 절대 추출하지 마십시오)
- TREATS: (Compound) -> (Disease) | 판정 기준: 약이 질병을 치료한다. 질병의 원인이나 진행 자체에 작용한다.
- PALLIATES: (Compound) -> (Disease) | 판정 기준: 약이 질병의 증상을 완화한다. 질병 자체는 그대로 두고 증상만 덜어 준다.
- BINDS: (Compound) -> (Gene) | 판정 기준: 약이 그 유전자의 단백질에 결합한다. 대사·수송을 맡는다는 진술도 포함한다.
- UPREGULATES_CG: (Compound) -> (Gene) | 판정 기준: 약이 그 유전자의 발현을 증가시킨다.
- DOWNREGULATES_CG: (Compound) -> (Gene) | 판정 기준: 약이 그 유전자의 발현을 감소시킨다.
- ASSOCIATES: (Disease) -> (Gene) | 판정 기준: 질병과 유전자 사이에 연관이 보고됐다.
- PRESENTS: (Disease) -> (Symptom) | 판정 기준: 질병이 그 증상으로 나타난다.
- INCLUDES: (PharmacologicClass) -> (Compound) | 판정 기준: 약효 분류가 그 약물을 포함한다 (계열 소속 관계).
- RESEMBLES_DD: (Disease) -> (Disease) | 판정 기준: 두 질병의 임상 양상이나 발병 기전이 유사하다.


## 2. Pydantic 스키마: Literal Enum 제약 및 CoT 추론 컨테이너

In [3]:
RelationName = Literal[
    "TREATS", "PALLIATES", "BINDS", "UPREGULATES_CG", "DOWNREGULATES_CG",
    "ASSOCIATES", "PRESENTS", "INCLUDES", "RESEMBLES_DD"
]
NodeType = Literal["Compound", "Disease", "Gene", "PharmacologicClass", "Symptom"]

class Triple(BaseModel):
    subject: str = Field(description="주어 개체명")
    subject_type: NodeType = Field(description="주어 노드 타입")
    relation: RelationName = Field(description="온톨로지 허용 관계")
    object: str = Field(description="목적어 개체명")
    object_type: NodeType = Field(description="목적어 노드 타입")
    evidence: str = Field(description="원문 근거 문장")

class CoTExtraction(BaseModel):
    reasoning: str = Field(description="추론 과정 단계별 서술")
    triples: List[Triple] = Field(description="추출된 트리플 목록")

# 인스턴스화 테스트
sample = Triple(
    subject="Warfarin",
    subject_type="Compound",
    relation="BINDS",
    object="VKORC1",
    object_type="Gene",
    evidence="Warfarin exerts its anticoagulant effect by inhibiting VKORC1."
)
print('✅ [Pydantic 스키마 검증 통과] Triple 및 CoTExtraction 정상 동작')
print(f'• 인스턴스: {sample.subject} ({sample.subject_type}) --[{sample.relation}]--> {sample.object} ({sample.object_type})')

✅ [Pydantic 스키마 검증 통과] Triple 및 CoTExtraction 정상 동작
• 인스턴스: Warfarin (Compound) --[BINDS]--> VKORC1 (Gene)


## 3. 표준 식별자(ID) 조회 및 :Candidate 격리 로직

In [4]:
name2id_file = Path('data/name2id.json')
with open(name2id_file, 'r', encoding='utf-8') as f:
    name2id = json.load(f)

def lookup_id(name: str, name2id: dict) -> Tuple[Optional[str], str]:
    cleaned = name.strip()
    val = name2id.get(cleaned) or name2id.get(cleaned.lower())
    if val is None:
        return None, "miss"
    if isinstance(val, list):
        return None, "ambiguous"
    return str(val), "hit"

print(f"✅ [사전 로드 완료] 총 {len(name2id):,}개 엔트리 바인딩")
for test_name in ["Warfarin", "VKORC1", "UnknownDrug"]:
    nid, reason = lookup_id(test_name, name2id)
    print(f"• {test_name} 조회: ID={nid}, 사유={reason}")

✅ [사전 로드 완료] 총 18,838개 엔트리 바인딩
• Warfarin 조회: ID=DB00682, 사유=hit
• VKORC1 조회: ID=7900, 사유=hit
• UnknownDrug 조회: ID=None, 사유=miss


## 4. 3단계 무결성 정제 퍼널 (Triple Refinement Funnel)

In [5]:
def ground_check(evidence: str, raw_text: str, subject: str, object_: str) -> bool:
    if not evidence or evidence not in raw_text:
        return False
    ev_lower = evidence.lower()
    return (subject.lower() in ev_lower) and (object_.lower() in ev_lower)

def check_signature(row: dict, signatures: dict = RELATION_SIGNATURES) -> bool:
    rel = row.get("relation")
    if rel not in signatures:
        return False
    subj_t, obj_t, _ = signatures[rel]
    return (row.get("subject_type") == subj_t) and (row.get("object_type") == obj_t)

def drop_duplicates(rows: List[dict]) -> List[dict]:
    seen = set()
    unique = []
    for r in rows:
        key = (r["subject"].lower().strip(), r["relation"], r["object"].lower().strip())
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique

# LV1 데이터 대상 테스트
lv1_file = Path('data/pgx_lv1_triples.jsonl')
raw_triples = []
with open(lv1_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            raw_triples.append(json.loads(line))

valid_sig = [t for t in raw_triples if check_signature(t)]
clean_triples = drop_duplicates(valid_sig)

print(f"• 원시 입력 건수: {len(raw_triples)}건")
print(f"• 1단계 시그니처 검사 통과: {len(valid_sig)}건 (기각: {len(raw_triples) - len(valid_sig)}건)")
print(f"• 2단계 복합키 중복 제거 통과: {len(clean_triples)}건 (제거: {len(valid_sig) - len(clean_triples)}건)")
print(f"✅ [정제 퍼널 통과 완료] 최종 {len(clean_triples)}건의 무결점 트리플 확보")

• 원시 입력 건수: 12건
• 1단계 시그니처 검사 통과: 10건 (기각: 2건)
• 2단계 복합키 중복 제거 통과: 9건 (제거: 1건)
✅ [정제 퍼널 통과 완료] 최종 9건의 무결점 트리플 확보


## 5. 타입 계층(`TYPE_HIERARCHY`) 및 Neo4j Cypher MERGE 적재

In [6]:
TYPE_HIERARCHY = {
    "Enzyme": "Gene",
    "Transporter": "Gene",
    "Cytokine": "Gene",
    "Autoimmune": "Disease",
}

def labels_for(node_type: str) -> List[str]:
    parent = TYPE_HIERARCHY.get(node_type)
    return [parent, node_type] if parent else [node_type]

def node_pattern(var: str, name: str, node_type: str, name2id: dict) -> Tuple[str, dict]:
    nid, _ = lookup_id(name, name2id)
    if nid:
        return f"MERGE ({var}:{node_type} {{id: ${var}_id}})\nON CREATE SET {var}.name = ${var}_name", {f"{var}_id": nid, f"{var}_name": name}
    else:
        return f"MERGE ({var}:Candidate {{name: ${var}_name}})", {f"{var}_name": name}

def triple_to_cypher(row: dict, name2id: dict, doc_id: str = "DOC_01", evidence_level: str = "reported") -> Tuple[str, dict]:
    s_c, s_p = node_pattern("s", row["subject"], row["subject_type"], name2id)
    o_c, o_p = node_pattern("o", row["object"], row["object_type"], name2id)
    rel = row["relation"]
    r_c = (
        f"MERGE (s)-[r:{rel}]->(o)\n"
        f"ON CREATE SET r.evidence = $evidence, r.doc_id = $doc_id, r.evidence_level = $evidence_level, r.created_at = datetime()\n"
        f"ON MATCH SET r.evidence_level = CASE WHEN r.evidence_level = 'curated' THEN 'curated' ELSE $evidence_level END"
    )
    params = {**s_p, **o_p, "evidence": row.get("evidence", ""), "doc_id": doc_id, "evidence_level": evidence_level}
    return f"{s_c}\n{o_c}\n{r_c};", params

print(f"• Enzyme 다중 레이블: {labels_for('Enzyme')}")
print(f"• Disease 다중 레이블: {labels_for('Disease')}")

test_row = clean_triples[0]
cypher, params = triple_to_cypher(test_row, name2id, doc_id="PMC13494166")
print("--- [생성된 파라미터화 Cypher MERGE 문] ---")
print(cypher)
print("--- [바인딩 파라미터] ---")
print(json.dumps(params, ensure_ascii=False, indent=2))
print("✅ [PASS] 멱등 Cypher 및 파라미터 생성 무결성 검증 완료")

• Enzyme 다중 레이블: ['Gene', 'Enzyme']
• Disease 다중 레이블: ['Disease']
--- [생성된 파라미터화 Cypher MERGE 문] ---
MERGE (s:Compound {id: $s_id})
ON CREATE SET s.name = $s_name
MERGE (o:Gene {id: $o_id})
ON CREATE SET o.name = $o_name
MERGE (s)-[r:BINDS]->(o)
ON CREATE SET r.evidence = $evidence, r.doc_id = $doc_id, r.evidence_level = $evidence_level, r.created_at = datetime()
ON MATCH SET r.evidence_level = CASE WHEN r.evidence_level = 'curated' THEN 'curated' ELSE $evidence_level END;
--- [바인딩 파라미터] ---
{
  "s_id": "DB00682",
  "s_name": "Warfarin",
  "o_id": "7900",
  "o_name": "VKORC1",
  "evidence": "Warfarin inhibits VKORC1 subunit 1.",
  "doc_id": "PMC13494166",
  "evidence_level": "reported"
}
✅ [PASS] 멱등 Cypher 및 파라미터 생성 무결성 검증 완료


## 6. 결론 및 종합 요약

- **온톨로지는 지식그래프의 계약서**: 8+1대 시그니처와 `Literal` 스키마 제약으로 LLM의 허용 범위를 통제하고 관계 파편화를 방지했습니다.
- **식별자 기반 무결성 보장**: `name2id.json` 매핑을 통해 동음이의어와 미등록어를 구분하고, 미등록 개체는 `:Candidate`로 안전하게 격리했습니다.
- **3단계 정제 퍼널**: 원문 대조, 시그니처 정합성, 중복 제거를 통해 환각 및 방향 오류를 원천 차단했습니다.
- **프로덕션 연동**: n8n 및 자동화 스케줄러를 통해 이 파이프라인을 실서비스 지식그래프(DART-Trace 등)에 즉각 적용할 수 있습니다.